# Phase 4: Time-Series Forecasting with Prophet
## Industrial Demand Forecasting and Sales Analytics System

This notebook demonstrates fitting the **Facebook Prophet** model on the alloy sales data. Prophet is selected due to:
- Robustness to daily missing data / irregular gaps.
- Ability to handle B2B weekly purchase cycles and quarterly/annual seasonality.
- Provision of explicit confidence intervals.

In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

### 1. Load Data
We read in `data/sales_data.csv` and group it by product and region.

In [ ]:
data_path = os.path.join("..", "data", "sales_data.csv")
df = pd.read_csv(data_path)
df['Order_Date'] = pd.to_datetime(df['Order_Date'])

# Aggregate daily sales per product-region series
ts_data = df.groupby(['Order_Date', 'Product_Code', 'Region'])['Quantity'].sum().reset_index()
print(f"Time series dataset contains {ts_data.shape[0]} daily records.")
ts_data.head()

### 2. Fit Prophet on a Single Product-Region Combination
Let's select a single series to visualize its fit and future 90-day forecast.

In [ ]:
prod = ts_data['Product_Code'].unique()[0]
region = ts_data['Region'].unique()[0]
print(f"Selected series: Product {prod} in Region {region}")

single_series = ts_data[(ts_data['Product_Code'] == prod) & (ts_data['Region'] == region)].copy()

# Fit Prophet required format
prophet_df = single_series[['Order_Date', 'Quantity']].rename(columns={'Order_Date': 'ds', 'Quantity': 'y'})

# Split train/test (last 90 days as test)
split_date = prophet_df['ds'].max() - pd.Timedelta(days=90)
train = prophet_df[prophet_df['ds'] <= split_date]
test = prophet_df[prophet_df['ds'] > split_date]

# Fit model
m = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
m.fit(train)

# Forecast
future = m.make_future_dataframe(periods=90, freq='D')
forecast = m.predict(future)

# Plot forecast
fig1 = m.plot(forecast)
plt.title(f"Prophet Forecast for {prod} ({region})")
plt.show()

### 3. Display Seasonality Components
Prophet allows us to decompose the trend, weekly cycles, and yearly seasonality patterns.

In [ ]:
fig2 = m.plot_components(forecast)
plt.show()